In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset ,DataLoader
import spacy
from collections import Counter
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
print(torch.cuda.get_device_name(0))
print(torch.__version__)

cuda
Tesla T4
2.10.0+cu128


## loading data

In [2]:
def load_data():
    data_path = "/kaggle/input/datasets/vivekmettu/wikitext2-data"
    train_data_path = data_path + "/train.txt"
    test_data_path = data_path + "/test.txt"
    with open(train_data_path, "r", encoding="utf-8") as f:
        train_text = f.read()
    with open(test_data_path,"r",encoding="utf-8") as f:
        test_text = f.read()
    return train_text,test_text    
train_data,test_data = load_data()

## vocabulary

In [3]:
nlp = spacy.load("en_core_web_sm",disable=["parser", "ner"])

def vocabulary(train_data,test_data):
    train_tokens = []
    test_tokens = []
    for doc in nlp.pipe(train_data.splitlines(), batch_size=1000):
        train_tokens.extend([token.text.lower() for token in doc if not token.is_space])

    for doc in nlp.pipe(test_data.splitlines(), batch_size=1000):
        test_tokens.extend([token.text.lower() for token in doc if not token.is_space])  

    vocab_counter = Counter(train_tokens)
    vocab = {"<PAD>":0,"<UNK>":1}
    vocab.update({word:idx+2 for idx,(word,_) in enumerate(vocab_counter.items())})

    return train_tokens,test_tokens,vocab

train_tokens, test_tokens, vocab = vocabulary(train_data,test_data)

print(train_tokens[:10])
print(test_tokens[:10])
print(list(vocab.items())[:10])

['"', '=', '2013', '–', '14', 'york', 'city', 'f.c.', 'season', '=']
['"', '=', 'tropical', 'storm', '<', 'unk', '>', '(', '2008', ')']
[('<PAD>', 0), ('<UNK>', 1), ('"', 2), ('=', 3), ('2013', 4), ('–', 5), ('14', 6), ('york', 7), ('city', 8), ('f.c.', 9)]


## Encoding

In [4]:
def encode_tokens(tokens, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]

train_encoded = encode_tokens(train_tokens, vocab)
test_encoded = encode_tokens(test_tokens, vocab)

## sequence


In [5]:
def create_sequences(encoded_tokens, seq_length):
    sequences = []
    labels = []
    for i in range(seq_length,len(encoded_tokens)):
        seq = encoded_tokens[i-seq_length:i]
        label = encoded_tokens[i]
        sequences.append(seq)
        labels.append(label)
    return sequences, labels

train_sequences, train_labels = create_sequences(train_encoded, seq_length=5)

## dataloader

In [6]:
x_train,y_train = torch.tensor(train_sequences,dtype=torch.long),torch.tensor(train_labels,dtype=torch.long)
train_dataset = TensorDataset(x_train,y_train)
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)

## LSTM + GRU HYBRID MODEL

In [7]:
class Hybrid_model(nn.Module):
    def __init__(self, vocab_size,embedded_dim = 64,hidden_dim = 128):
        super(Hybrid_model,self).__init__()

        self.embedding = nn.Embedding(vocab_size,embedded_dim)
        self.lstm = nn.LSTM(embedded_dim,hidden_dim,batch_first=True)
        self.gru = nn.GRU(embedded_dim,hidden_dim,batch_first=True)

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim*2,64),
            nn.ReLU(),
            nn.Linear(64,vocab_size)
        )

    def forward(self,x):
        x = self.embedding(x) 
        lstm_out,_ = self.lstm(x)
        gru_out,_ = self.gru(x)   
        lstm_last = lstm_out[:,-1,:]
        gru_last = gru_out[:,-1,:]
        x = torch.cat((lstm_last, gru_last), dim=-1)
        x = self.fc(x)
        return x

## criterion and optimizer

In [8]:
model = Hybrid_model(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Hybrid_model(
  (embedding): Embedding(28868, 64)
  (lstm): LSTM(64, 128, batch_first=True)
  (gru): GRU(64, 128, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=28868, bias=True)
  )
)
Total parameters: 3914244
Trainable parameters: 3914244


In [9]:
for name, param in model.lstm.named_parameters():
    print(name, param.shape)

for name, param in model.gru.named_parameters():
    print(name, param.shape)

weight_ih_l0 torch.Size([512, 64])
weight_hh_l0 torch.Size([512, 128])
bias_ih_l0 torch.Size([512])
bias_hh_l0 torch.Size([512])
weight_ih_l0 torch.Size([384, 64])
weight_hh_l0 torch.Size([384, 128])
bias_ih_l0 torch.Size([384])
bias_hh_l0 torch.Size([384])


## training loop

In [20]:
def train_model(model,train_loader,criterion,optimizer,device,epochs = 10,check_point_path = "checkpoint.pth"):
    model.to(device)
    start_epoch = 0
    try:
        checkpoint = torch.load(check_point_path)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            print(f"Resuming from epoch {start_epoch}")

        else:
            model.load_state_dict(checkpoint)
            print("Loaded weights only, starting fresh optimizer.")   
    except FileNotFoundError:
        print("No checkpoint found, starting fresh.")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for x_train,y_train in train_loader:
            x_train,y_train = x_train.to(device),y_train.to(device)
            optimizer.zero_grad()
            outputs = model(x_train)
            loss = criterion(outputs,y_train)
            loss.backward()
            optimizer.step()

            total_train_loss +=loss.item()
        avg_train_loss = total_train_loss/len(train_loader)

        print(f"Epoch {epoch+1}/{epochs}, "
              f"Train Loss: {avg_train_loss:.4f}, ")

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
        }, "check_point.pth")

check_point_path = "/kaggle/input/datasets/kaushikrajagiri/checkpoint-pth/check_point.pth"
train_model(model=model,train_loader=train_loader,criterion=criterion,optimizer=optimizer,device=device,check_point_path=check_point_path)


Loaded weights only, starting fresh optimizer.
Epoch 1/10, Train Loss: 5.2999, 
Epoch 2/10, Train Loss: 5.2901, 
Epoch 3/10, Train Loss: 5.2794, 
Epoch 4/10, Train Loss: 5.2696, 
Epoch 5/10, Train Loss: 5.2611, 
Epoch 6/10, Train Loss: 5.2534, 
Epoch 7/10, Train Loss: 5.2467, 
Epoch 8/10, Train Loss: 5.2404, 
Epoch 9/10, Train Loss: 5.2339, 
Epoch 10/10, Train Loss: 5.2285, 


In [24]:
trained_params_path = "/kaggle/input/datasets/kaushikrajagiri/checkpoint1-pth/check_point (1).pth"
trained_params = torch.load(trained_params_path,map_location=device)
if isinstance(trained_params,dict) and 'model_state_dict' in trained_params:
    model.load_state_dict(trained_params['model_state_dict'])
    print(f"loadod mode_state_dict (epoch {trained_params.get('epoch', '?')},"
          f"loss {trained_params.get('loss', '?')})")

else:
    model.load_state_dict(trained_params)
    print("loaded raw state dict")    

model.to(device)
model.eval()    

loadod mode_state_dict (epoch 9,loss 5.228450947721955)


Hybrid_model(
  (embedding): Embedding(28868, 64)
  (lstm): LSTM(64, 128, batch_first=True)
  (gru): GRU(64, 128, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=28868, bias=True)
  )
)

## test dataloader

In [25]:
seq = 5
test_seqs,test_labels = create_sequences(test_encoded,seq_length=seq)
x_test,y_test = torch.tensor(test_seqs,dtype=torch.long),torch.tensor(test_labels,dtype=torch.long)
test_dataset = TensorDataset(x_test,y_test)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)
print(f"Test sequences: {len(test_dataset)}")

Test sequences: 240179


## Evaluation

In [33]:
import math

@torch.no_grad()
def evaluate_loss(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for x_batch, y_batch in data_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        total_loss += loss.item() * x_batch.size(0)
        total_tokens += x_batch.size(0)
    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

test_loss, test_ppl = evaluate_loss(model, test_loader, criterion, device)
print(f"Test loss: {test_loss:.4f}")
print(f"Test perplexity: {test_ppl:.2f}")


Test loss: 5.2830
Test perplexity: 196.96


In [32]:
@torch.no_grad()
def evaluate_accuracy(model, data_loader, device, topk=(1, 5)):
    model.eval()
    max_k = max(topk)
    total = 0
    correct = {k: 0 for k in topk}

    for x_batch, y_batch in data_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        outputs = model(x_batch)  # (batch, vocab_size)
        _, top_preds = outputs.topk(max_k, dim=1)  # (batch, max_k)

        for k in topk:
            match = (top_preds[:, :k] == y_batch.unsqueeze(1)).any(dim=1)
            correct[k] += match.sum().item()

        total += y_batch.size(0)

    return {k: correct[k] / total for k in topk}

acc = evaluate_accuracy(model, test_loader, device, topk=(1, 5))
for k, v in acc.items():
    print(f"Top-{k} accuracy: {v*100:.2f}%")


Top-1 accuracy: 27.99%
Top-5 accuracy: 44.05%


In [41]:
idx2word = {idx: word for word, idx in vocab.items()}

@torch.no_grad()
def generate_text(model, seed_text, vocab, idx2word, seq_length=5,
                   num_words=20, device=device, temperature=0.5):
    model.eval()
    tokens = [t.text.lower() for t in nlp(seed_text) if not t.is_space]
    generated = tokens.copy()

    for _ in range(num_words):
        context = generated[-seq_length:]
        if len(context) < seq_length:
            context = ["<PAD>"] * (seq_length - len(context)) + context
        encoded = torch.tensor(
            [[vocab.get(w, vocab["<UNK>"]) for w in context]],
            dtype=torch.long, device=device
        )
        logits = model(encoded).squeeze(0) / temperature
        probs = torch.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1).item()
        generated.append(idx2word.get(next_idx, "<UNK>"))

    return " ".join(generated)

for seed in ["who are "]:
    print(f"Seed: {seed!r}")
    print(" ->", generate_text(model, seed, vocab, idx2word, seq_length=seq, num_words=15))
    print()


Seed: 'who are '
 -> who are said to be the < unk > to the republic of the < unk >

